# BankRisk Compass: Credit Default Risk Modeling

This notebook builds a credit-risk model that estimates the probability that a loan applicant will default. The goal is not only to maximize model scores, but to make the decision logic useful for a bank: missed defaults are expensive, while unnecessary rejections can hurt revenue and customer experience.

The productionized pieces of this project live outside the notebook:

- `src/train_model.py` contains the leakage-safe training pipeline.
- `models/credit_risk_model.pkl` stores the final model bundle.
- `reports/` stores model metrics, charts, threshold analysis, and feature importance.
- `app/views.py` and `app/templates/` provide the Django dashboard.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "credit_risk.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODEL_PATH = PROJECT_ROOT / "models" / "credit_risk_model.pkl"

sns.set_theme(style="whitegrid", palette="Set2")

## 1. Load the Dataset

Each row represents a loan applicant. The target variable is `loan_status`, where `1` means default and `0` means non-default.

In [ ]:
data = pd.read_csv(DATA_PATH)
print(data.shape)
data.head()

In [ ]:
data.info()

In [ ]:
data.describe(include="all").T

## 2. Data Quality Checks

The model code handles missing values inside the scikit-learn pipeline. That matters because imputing before the train/test split would leak information from the test set into training.

In [ ]:
quality = pd.DataFrame({
    "missing_values": data.isna().sum(),
    "missing_rate": data.isna().mean(),
    "unique_values": data.nunique()
}).sort_values("missing_values", ascending=False)
quality

In [ ]:
class_balance = data["loan_status"].value_counts(normalize=True).rename({0: "non_default", 1: "default"})
class_balance

Interpretation: the data is imbalanced. Most borrowers do not default, so accuracy alone is not enough. Recall, precision, F1-score, ROC-AUC, and business thresholding are more informative.

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="loan_status")
plt.title("Default vs Non-default Distribution")
plt.xlabel("Loan status")
plt.ylabel("Applicants")
plt.show()

The target distribution confirms the class imbalance: non-defaults dominate the dataset.

In [ ]:
numeric_cols = data.select_dtypes(include="number").columns.tolist()
data[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Numeric Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=data, x="loan_status", y="person_income", ax=axes[0])
axes[0].set_ylim(0, 200000)
axes[0].set_title("Income by Loan Status")

sns.boxplot(data=data, x="loan_status", y="loan_amnt", ax=axes[1])
axes[1].set_title("Loan Amount by Loan Status")

sns.boxplot(data=data, x="loan_status", y="loan_int_rate", ax=axes[2])
axes[2].set_title("Interest Rate by Loan Status")

plt.tight_layout()
plt.show()

Defaulted loans tend to show higher interest rates and heavier loan burden relative to income. Income has strong outliers, so the chart is capped to keep the central pattern visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

grade_default = data.groupby("loan_grade")["loan_status"].mean().sort_index()
sns.barplot(x=grade_default.index, y=grade_default.values, ax=axes[0])
axes[0].set_title("Default Rate by Loan Grade")
axes[0].set_ylabel("Default rate")
axes[0].set_xlabel("Loan grade")

intent_default = data.groupby("loan_intent")["loan_status"].mean().sort_values(ascending=False)
sns.barplot(x=intent_default.values, y=intent_default.index, ax=axes[1])
axes[1].set_title("Default Rate by Loan Intent")
axes[1].set_xlabel("Default rate")
axes[1].set_ylabel("Loan intent")

plt.tight_layout()
plt.show()

Loan grade is one of the clearest risk separators. Loan intent also contains useful signal, especially when combined with borrower income and loan burden.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(data[numeric_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation Between Numeric Features")
plt.tight_layout()
plt.show()

## 4. Leakage-Safe Modeling Workflow

Preprocessing is fitted only on the training split using `ColumnTransformer` and `Pipeline`:

- numeric features: median imputation and scaling
- categorical features: most-frequent imputation and one-hot encoding
- model: Random Forest selected after comparison with Logistic Regression and Gradient Boosting

In [ ]:
from src.train_model import (
    CATEGORICAL_FEATURES,
    FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    build_preprocessor,
)

print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
build_preprocessor()

## 5. Train or Load Project Artifacts

The training script compares models, tunes a Random Forest, tunes a business threshold, saves the model, and writes report files. If artifacts already exist, this cell simply reads them.

In [ ]:
from src.train_model import train_and_save

if not MODEL_PATH.exists():
    train_and_save(quick=True)

model_comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv")
final_metrics = pd.read_csv(REPORTS_DIR / "final_model_metrics.csv")
threshold_analysis = pd.read_csv(REPORTS_DIR / "threshold_analysis.csv")
permutation_importance = pd.read_csv(REPORTS_DIR / "permutation_importance.csv")

model_comparison

The Random Forest gives the strongest balance of precision, recall, F1-score, and ROC-AUC on the test set.

In [ ]:
model_comparison.set_index("model")[["accuracy", "precision", "recall", "f1_score", "roc_auc"]].plot(
    kind="bar",
    figsize=(11, 5),
    ylim=(0, 1),
    rot=0,
)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("")
plt.tight_layout()
plt.show()

## 6. Business-Focused Threshold Tuning

A normal `0.50` threshold is not always the best business decision. In this project, a false negative is treated as 5x more costly than a false positive:

- False negative: a risky borrower is approved.
- False positive: a safer borrower is rejected or sent to manual review.

In [ ]:
final_metrics

In [ ]:
selected_threshold = final_metrics.loc[final_metrics["decision_threshold"] != 0.5, "decision_threshold"].iloc[0]

plt.figure(figsize=(11, 5))
plt.plot(threshold_analysis["threshold"], threshold_analysis["precision"], label="Precision")
plt.plot(threshold_analysis["threshold"], threshold_analysis["recall"], label="Recall")
plt.plot(threshold_analysis["threshold"], threshold_analysis["f1_score"], label="F1-score")
plt.axvline(selected_threshold, color="black", linestyle="--", label=f"Business threshold = {selected_threshold:.2f}")
plt.title("Threshold Tradeoff")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

The business threshold increases default recall by accepting more manual review or rejection cases. This is more realistic for credit risk than optimizing accuracy alone.

## 7. Model Explanation

Permutation importance is used to explain which original applicant features matter most to the final model. This avoids relying only on one-hot encoded internal feature names.

In [ ]:
top_importance = permutation_importance.head(10).sort_values("importance_mean")
plt.figure(figsize=(10, 6))
plt.barh(top_importance["feature"], top_importance["importance_mean"])
plt.title("Top Credit Risk Drivers")
plt.xlabel("Permutation importance")
plt.tight_layout()
plt.show()

permutation_importance.head(10)

## 8. Final Report

**Best model:** Random Forest

**Default threshold results:**

- Accuracy: 93.4%
- F1-score: 0.825
- Recall for defaults: 0.715
- ROC-AUC: 0.931

**Business threshold results:**

- Selected threshold: 0.26
- Accuracy: 91.3%
- F1-score: 0.797
- Recall for defaults: 0.784

**Business interpretation:** the Random Forest performs well at separating safer borrowers from higher-risk borrowers. At the standard 0.50 threshold it reaches about 93% accuracy and an F1-score around 0.82. For a bank, however, the threshold should reflect business costs. The lower business threshold catches more potential defaults, reducing the chance that a risky borrower is approved, while increasing the number of applicants sent to review.

The saved model is available at `models/credit_risk_model.pkl`, and the Django dashboard runs with `python manage.py runserver`.